# Plantilla NLP - Examen IA
## Ejemplos de Modelos de Procesamiento de Lenguaje Natural

Este notebook contiene ejemplos prácticos de modelos NLP con **diferentes configuraciones** para entender:
- Vectorización (TF-IDF, Count Vectorizer, Word Embeddings)
- Manejo de palabras desconocidas (OOV - Out Of Vocabulary)
- Embeddings (pre-entrenados vs. entrenables)
- Capas de salida según el problema
- Mejoras y optimizaciones del modelo

## 1. Imports y Configuración Inicial

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Configurar semilla para reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

## 2. Dataset de Ejemplo
Creamos un dataset simple de clasificación de sentimientos (positivo/negativo)

In [ ]:
# Dataset de ejemplo - Clasificación binaria de sentimientos
texts = [
    "Esta película es increíble y muy emocionante",
    "Me encanta este producto, es excelente",
    "Qué gran experiencia, totalmente recomendado",
    "Fantástico servicio, muy satisfecho",
    "Horrible, no lo recomiendo para nada",
    "Muy malo, pérdida de tiempo y dinero",
    "Decepcionante, esperaba mucho más",
    "Pésima calidad, no vale la pena",
    "Absolutamente maravilloso, me fascina",
    "Terrible experiencia, muy insatisfecho",
    "Excelente calidad y buen precio",
    "No funciona bien, muy frustrante",
    "Perfecto, justo lo que necesitaba",
    "Deficiente y caro, no lo compraría de nuevo",
    "Increíble producto, superó mis expectativas"
]

# 1 = positivo, 0 = negativo
labels = [1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1]

print(f"Total de textos: {len(texts)}")
print(f"Total de etiquetas: {len(labels)}")
print(f"\nEjemplo de texto: '{texts[0]}'")
print(f"Etiqueta: {labels[0]} (1=positivo, 0=negativo)")

## 3. VECTORIZACIÓN - Tokenizer

### Parámetros Importantes del Tokenizer:
- **`num_words`**: Limita el vocabulario a las N palabras más frecuentes
- **`oov_token`**: Token para palabras desconocidas (Out Of Vocabulary)
- **`lower`**: Convierte todo a minúsculas
- **`char_level`**: Si True, tokeniza por caracteres en vez de palabras

In [ ]:
# EJEMPLO 1: Tokenizer SIN oov_token
vocab_size = 100
max_length = 20

tokenizer_sin_oov = Tokenizer(num_words=vocab_size)
tokenizer_sin_oov.fit_on_texts(texts)

print("=" * 60)
print("TOKENIZER SIN OOV_TOKEN")
print("=" * 60)
print(f"Tamaño del vocabulario: {len(tokenizer_sin_oov.word_index)}")
print(f"\nPrimeras 10 palabras del vocabulario:")
for word, idx in list(tokenizer_sin_oov.word_index.items())[:10]:
    print(f"  '{word}': {idx}")

# Probar con una frase que contiene palabras nuevas
test_sentence = ["Esta película es horrible y aburrida"]
sequences = tokenizer_sin_oov.texts_to_sequences(test_sentence)
print(f"\nFrase de prueba: '{test_sentence[0]}'")
print(f"Secuencia resultante: {sequences[0]}")
print("PROBLEMA: Las palabras 'aburrida' no están en el vocabulario y se ignoran!")

In [ ]:
# EJEMPLO 2: Tokenizer CON oov_token (RECOMENDADO)
tokenizer_con_oov = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer_con_oov.fit_on_texts(texts)

print("\n" + "=" * 60)
print("TOKENIZER CON OOV_TOKEN")
print("=" * 60)
print(f"Tamaño del vocabulario: {len(tokenizer_con_oov.word_index)}")
print(f"\nToken OOV: '{tokenizer_con_oov.oov_token}' -> índice: {tokenizer_con_oov.word_index['<OOV>']}")

# Probar con la misma frase
sequences_oov = tokenizer_con_oov.texts_to_sequences(test_sentence)
print(f"\nFrase de prueba: '{test_sentence[0]}'")
print(f"Secuencia resultante: {sequences_oov[0]}")
print("MEJOR: Las palabras desconocidas se representan con el token <OOV> (índice 1)")

# Convertir secuencias a texto
padded_sequences = pad_sequences(sequences_oov, maxlen=max_length, padding='post')
print(f"\nSecuencia con padding: {padded_sequences[0]}")

### Preparar Datos para Entrenamiento

In [ ]:
# Usar el tokenizer con OOV para todo el dataset
sequences = tokenizer_con_oov.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

# Convertir a arrays de numpy
X = np.array(padded_sequences)
y = np.array(labels)

# Split de datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape de X_train: {X_train.shape}")
print(f"Shape de y_train: {y_train.shape}")
print(f"Shape de X_test: {X_test.shape}")
print(f"Shape de y_test: {y_test.shape}")
print(f"\nEjemplo de secuencia procesada:")
print(f"Texto original: '{texts[0]}'")
print(f"Secuencia: {X[0]}")

## 4. MODELOS DE NLP - Diferentes Arquitecturas

---

### MODELO 1: Embeddings + Flatten (BÁSICO)

In [ ]:
"""
MODELO 1: Embeddings + Flatten
================================
Ventajas:
   - Simple y rápido
   - Pocas capas, menos overfitting en datasets pequeños
   
Desventajas:
   - No captura relaciones secuenciales
   - Pierde información del orden de las palabras
   - Necesita muchos parámetros si el vocabulario es grande
"""

embedding_dim = 16

model_1 = keras.Sequential([
    # Capa de Embedding: convierte índices de palabras en vectores densos
    layers.Embedding(input_dim=vocab_size, 
                    output_dim=embedding_dim, 
                    input_length=max_length,
                    name='embedding'),
    
    # Flatten: convierte la matriz 3D en 2D
    # Shape: (batch, max_length, embedding_dim) -> (batch, max_length * embedding_dim)
    layers.Flatten(name='flatten'),
    
    # Capa densa oculta
    layers.Dense(16, activation='relu', name='hidden'),
    
    # CAPA DE SALIDA para clasificación binaria
    # sigmoid: salida entre 0 y 1
    layers.Dense(1, activation='sigmoid', name='output')
], name='Model_Embeddings_Flatten')

model_1.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # Para clasificación binaria
    metrics=['accuracy']
)

model_1.summary()

### MODELO 2: Embeddings + Global Average Pooling (MEJOR)

In [ ]:
"""
MODELO 2: Embeddings + GlobalAveragePooling1D
==============================================
Ventajas:
   - MENOS parámetros que Flatten
   - Más robusto a diferentes longitudes de secuencia
   - Mejor generalización
   - Promedia los embeddings (reduce dimensionalidad)
   
Desventajas:
   - Sigue sin capturar orden secuencial completo
   - El promedio puede perder información importante
"""

model_2 = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, 
                    output_dim=embedding_dim, 
                    input_length=max_length,
                    name='embedding'),
    
    # GlobalAveragePooling1D: calcula el promedio de cada embedding
    # Shape: (batch, max_length, embedding_dim) -> (batch, embedding_dim)
    layers.GlobalAveragePooling1D(name='global_avg_pool'),
    
    layers.Dense(16, activation='relu', name='hidden'),
    layers.Dropout(0.5, name='dropout'),  # Regularización
    
    # CAPA DE SALIDA para clasificación binaria
    layers.Dense(1, activation='sigmoid', name='output')
], name='Model_Embeddings_GAP')

model_2.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_2.summary()

print("\nCOMPARACIÓN DE PARÁMETROS:")
print(f"Modelo 1 (Flatten): {model_1.count_params():,} parámetros")
print(f"Modelo 2 (GAP): {model_2.count_params():,} parámetros")
print(f"Reducción: {model_1.count_params() - model_2.count_params():,} parámetros")

### MODELO 3: LSTM - Captura Secuencias (AVANZADO)

In [ ]:
"""
MODELO 3: Embeddings + LSTM
============================
Ventajas:
   - CAPTURA el orden secuencial de las palabras
   - Entiende contexto a largo plazo
   - Ideal para textos largos y complejos
   - Mejores resultados en tareas complejas de NLP
   
Desventajas:
   - MÁS parámetros (más lento)
   - Requiere más datos para entrenar bien
   - Puede hacer overfitting en datasets pequeños
   - Más costoso computacionalmente
"""

model_3 = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, 
                    output_dim=embedding_dim, 
                    input_length=max_length,
                    name='embedding'),
    
    # LSTM: Long Short-Term Memory
    # return_sequences=False: solo devuelve el último estado (para clasificación)
    # return_sequences=True: devuelve todos los estados (para seq2seq)
    layers.LSTM(32, return_sequences=False, name='lstm'),
    
    layers.Dense(16, activation='relu', name='hidden'),
    layers.Dropout(0.5, name='dropout'),
    
    # CAPA DE SALIDA para clasificación binaria
    layers.Dense(1, activation='sigmoid', name='output')
], name='Model_Embeddings_LSTM')

model_3.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_3.summary()

### MODELO 4: Bidirectional LSTM (MUY AVANZADO)

In [ ]:
"""
MODELO 4: Embeddings + Bidirectional LSTM
==========================================
Ventajas:
   - MEJOR rendimiento: lee el texto en ambas direcciones
   - Captura contexto pasado Y futuro
   - Ideal para tareas donde el contexto completo importa
   
Desventajas:
   - EL DOBLE de parámetros que LSTM normal
   - MÁS lento de entrenar
   - Requiere MUCHOS datos
   - No se puede usar en tiempo real (necesita ver todo el texto)
"""

model_4 = keras.Sequential([
    layers.Embedding(input_dim=vocab_size, 
                    output_dim=embedding_dim, 
                    input_length=max_length,
                    name='embedding'),
    
    # Bidirectional LSTM: procesa en ambas direcciones
    layers.Bidirectional(layers.LSTM(32, return_sequences=False), name='bidirectional_lstm'),
    
    layers.Dense(16, activation='relu', name='hidden'),
    layers.Dropout(0.5, name='dropout'),
    
    # CAPA DE SALIDA para clasificación binaria
    layers.Dense(1, activation='sigmoid', name='output')
], name='Model_Bidirectional_LSTM')

model_4.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_4.summary()

print("\nCOMPARACIÓN TOTAL DE MODELOS:")
print(f"Modelo 1 (Flatten):        {model_1.count_params():6,} parámetros")
print(f"Modelo 2 (GAP):            {model_2.count_params():6,} parámetros")
print(f"Modelo 3 (LSTM):           {model_3.count_params():6,} parámetros")
print(f"Modelo 4 (Bidirectional):  {model_4.count_params():6,} parámetros")

## 5. CAPAS DE SALIDA - Según el Tipo de Problema

---

### Clasificación Binaria (2 clases: 0 o 1)

In [ ]:
"""
CLASIFICACIÓN BINARIA
======================
Ejemplo: Sentimiento (positivo/negativo), Spam/No spam

CONFIGURACIÓN CORRECTA:
   - Capa de salida: Dense(1, activation='sigmoid')
   - Loss: 'binary_crossentropy'
   - Salida: valores entre 0 y 1 (probabilidad)
   - Etiquetas: 0 o 1
"""

model_binary = keras.Sequential([
    layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    layers.GlobalAveragePooling1D(),
    layers.Dense(16, activation='relu'),
    
    # CAPA DE SALIDA BINARIA
    layers.Dense(1, activation='sigmoid', name='binary_output')
], name='Binary_Classification')

model_binary.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # Loss para binario
    metrics=['accuracy']
)

print("CLASIFICACIÓN BINARIA")
print("Capa de salida: Dense(1, activation='sigmoid')")
print("Loss: binary_crossentropy")
print("Salida: [0.0 a 1.0] -> < 0.5 = clase 0, >= 0.5 = clase 1")
print(f"\nShape de salida: (batch_size, 1)")
model_binary.summary()

### Clasificación Multiclase (3+ clases)

In [ ]:
"""
CLASIFICACIÓN MULTICLASE
=========================
Ejemplo: Categorías de noticias (deportes, política, tecnología, ...)

CONFIGURACIÓN CORRECTA:
   - Capa de salida: Dense(num_classes, activation='softmax')
   - Loss: 'sparse_categorical_crossentropy' (si etiquetas son enteros)
          'categorical_crossentropy' (si etiquetas son one-hot)
   - Salida: vector de probabilidades (suma = 1.0)
"""

num_classes = 5  # Ejemplo: 5 categorías

model_multiclass = keras.Sequential([
    layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    
    # CAPA DE SALIDA MULTICLASE
    layers.Dense(num_classes, activation='softmax', name='multiclass_output')
], name='Multiclass_Classification')

model_multiclass.compile(
    optimizer='adam',
    # Si etiquetas son [0, 1, 2, 3, 4] -> sparse_categorical_crossentropy
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("CLASIFICACIÓN MULTICLASE")
print(f"Capa de salida: Dense({num_classes}, activation='softmax')")
print("Loss: sparse_categorical_crossentropy (etiquetas enteras)")
print("     o categorical_crossentropy (etiquetas one-hot)")
print(f"Salida: vector de {num_classes} probabilidades (suma=1.0)")
print(f"\nShape de salida: (batch_size, {num_classes})")
model_multiclass.summary()

### Clasificación Multi-etiqueta (Multiple Labels)

In [ ]:
"""
CLASIFICACIÓN MULTI-ETIQUETA
=============================
Ejemplo: Un texto puede ser [deportes=1, política=0, tecnología=1]
         (puede pertenecer a MÚLTIPLES categorías simultáneamente)

CONFIGURACIÓN CORRECTA:
   - Capa de salida: Dense(num_labels, activation='sigmoid')
   - Loss: 'binary_crossentropy'
   - Salida: cada etiqueta tiene su propia probabilidad [0-1]
   - Etiquetas: vector binario [1, 0, 1, 0, 1]
"""

num_labels = 5  # Ejemplo: 5 etiquetas posibles

model_multilabel = keras.Sequential([
    layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    
    # CAPA DE SALIDA MULTI-ETIQUETA
    # Sigmoid para CADA etiqueta independientemente
    layers.Dense(num_labels, activation='sigmoid', name='multilabel_output')
], name='Multilabel_Classification')

model_multilabel.compile(
    optimizer='adam',
    # binary_crossentropy se aplica a CADA etiqueta
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("CLASIFICACIÓN MULTI-ETIQUETA")
print(f"Capa de salida: Dense({num_labels}, activation='sigmoid')")
print("Loss: binary_crossentropy")
print(f"Salida: {num_labels} valores independientes [0.0-1.0]")
print("Cada salida es la probabilidad de que esa etiqueta esté presente")
print(f"\nShape de salida: (batch_size, {num_labels})")
model_multilabel.summary()

### Regresión (Predecir valores continuos)

In [ ]:
"""
REGRESIÓN
=========
Ejemplo: Predecir rating de una review (0.0 a 5.0 estrellas)

CONFIGURACIÓN CORRECTA:
   - Capa de salida: Dense(1, activation=None) o Dense(1, activation='linear')
   - Loss: 'mse' (Mean Squared Error) o 'mae' (Mean Absolute Error)
   - Salida: un valor continuo
"""

model_regression = keras.Sequential([
    layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    layers.GlobalAveragePooling1D(),
    layers.Dense(32, activation='relu'),
    
    # CAPA DE SALIDA REGRESIÓN
    # Sin activación (o 'linear') para valores continuos
    layers.Dense(1, activation='linear', name='regression_output')
], name='Regression')

model_regression.compile(
    optimizer='adam',
    loss='mse',  # Mean Squared Error
    metrics=['mae']  # Mean Absolute Error
)

print("REGRESIÓN")
print("Capa de salida: Dense(1, activation='linear')")
print("Loss: mse (Mean Squared Error) o mae (Mean Absolute Error)")
print("Salida: valor continuo (ej: 3.7 estrellas)")
print(f"\nShape de salida: (batch_size, 1)")
model_regression.summary()

## 6. TABLA RESUMEN - Capas de Salida

| Tipo de Problema | Capa de Salida | Función de Activación | Loss Function | Etiquetas |
|------------------|----------------|----------------------|---------------|-----------|
| **Clasificación Binaria** | `Dense(1)` | `sigmoid` | `binary_crossentropy` | `[0, 1]` |
| **Clasificación Multiclase** | `Dense(n_classes)` | `softmax` | `sparse_categorical_crossentropy` | `[0, 1, 2, ...]` |
| **Clasificación Multi-etiqueta** | `Dense(n_labels)` | `sigmoid` | `binary_crossentropy` | `[[0,1,1], [1,0,0], ...]` |
| **Regresión** | `Dense(1)` | `linear` o `None` | `mse` o `mae` | valores continuos |

## 7. ENTRENAR UN MODELO (Ejemplo con Modelo 2)

In [ ]:
# Entrenar el modelo 2 (Embeddings + GAP)
print("Entrenando Modelo 2 (GlobalAveragePooling)...")
print("=" * 60)

history = model_2.fit(
    X_train, y_train,
    epochs=50,
    batch_size=4,
    validation_data=(X_test, y_test),
    verbose=0  # Cambiar a 1 para ver progreso
)

print("Entrenamiento completado!")
print(f"\nAccuracy final en entrenamiento: {history.history['accuracy'][-1]:.4f}")
print(f"Accuracy final en validación: {history.history['val_accuracy'][-1]:.4f}")

### Visualizar Resultados del Entrenamiento

In [ ]:
# Gráficas de accuracy y loss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy durante el entrenamiento')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'], label='Train Loss', marker='o')
ax2.plot(history.history['val_loss'], label='Val Loss', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Loss durante el entrenamiento')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Hacer Predicciones

In [ ]:
# Probar con nuevas frases
new_texts = [
    "Este producto es maravilloso, lo recomiendo totalmente",
    "Muy decepcionado con la compra, no lo volvería a comprar",
    "Normal, ni bueno ni malo"
]

# Preprocesar
new_sequences = tokenizer_con_oov.texts_to_sequences(new_texts)
new_padded = pad_sequences(new_sequences, maxlen=max_length, padding='post')

# Predecir
predictions = model_2.predict(new_padded, verbose=0)

print("PREDICCIONES DEL MODELO:\n")
print("=" * 70)
for i, text in enumerate(new_texts):
    prob = predictions[i][0]
    sentiment = "POSITIVO" if prob >= 0.5 else "NEGATIVO"
    print(f"Texto: '{text}'")
    print(f"  -> Probabilidad: {prob:.4f}")
    print(f"  -> Predicción: {sentiment}")
    print("-" * 70)

## 8. MEJORAS Y OPTIMIZACIONES - ¿Qué cambiar?

### Preguntas Típicas de Examen y Respuestas

In [ ]:
"""
PREGUNTA 1: ¿Qué pasa si NO uso oov_token en el Tokenizer?
===============================================================
PROBLEMA:
   - Las palabras desconocidas se IGNORAN completamente
   - Pierdes información importante
   - Las secuencias pueden quedarse más cortas
   
SOLUCIÓN:
   - SIEMPRE usar oov_token="<OOV>"
   - El modelo aprenderá un embedding para palabras desconocidas


PREGUNTA 2: ¿Qué pasa si aumento embedding_dim?
===================================================
VENTAJAS:
   - Más capacidad para representar palabras
   - Puede capturar más relaciones semánticas
   
DESVENTAJAS:
   - MÁS parámetros (más lento, más memoria)
   - Mayor riesgo de overfitting si dataset es pequeño
   - Necesitas más datos para entrenar bien

RECOMENDACIÓN:
   - Datasets pequeños: 16-32 dimensiones
   - Datasets medianos: 64-128 dimensiones
   - Datasets grandes: 256-512 dimensiones


PREGUNTA 3: ¿Qué pasa si cambio Flatten por GlobalAveragePooling1D?
=======================================================================
MEJOR con GlobalAveragePooling1D:
   - Menos parámetros (más eficiente)
   - Mejor generalización
   - Menos overfitting
   - Funciona con secuencias de cualquier longitud
   
Flatten es peor porque:
   - Muchos más parámetros
   - Depende de max_length fijo
   - Más propenso a overfitting


PREGUNTA 4: ¿Cuándo usar LSTM vs GlobalAveragePooling?
==========================================================
Usa GlobalAveragePooling cuando:
   - Dataset pequeño
   - Textos cortos (tweets, títulos)
   - El orden de palabras no es crítico
   - Quieres rapidez
   
Usa LSTM cuando:
   - Dataset grande
   - Textos largos (artículos, reviews largas)
   - El orden y contexto son importantes
   - Tienes recursos computacionales


PREGUNTA 5: ¿Cómo mejorar el modelo si hace overfitting?
============================================================
1. Regularización:
   - Añadir Dropout(0.3-0.5) después de capas densas
   - Usar L2 regularization: Dense(units, kernel_regularizer=l2(0.001))

2. Reducir complejidad:
   - Menos capas
   - Menos neuronas por capa
   - Embedding_dim más pequeño

3. Más datos:
   - Data augmentation (sinónimos, traducción)
   - Técnicas de back-translation

4. Early Stopping:
   - Parar cuando val_loss deja de mejorar
   - callbacks=[EarlyStopping(patience=5)]

5. Batch Normalization:
   - Normalizar activaciones entre capas


PREGUNTA 6: ¿Qué pasa si cambio la capa de salida?
======================================================
CLASIFICACIÓN BINARIA:
   Dense(1, activation='sigmoid') + binary_crossentropy
   Salida: [0.0 a 1.0]

CLASIFICACIÓN MULTICLASE:
   Dense(n_classes, activation='softmax') + sparse_categorical_crossentropy
   Salida: [p1, p2, p3, ...] suma = 1.0

MULTI-ETIQUETA:
   Dense(n_labels, activation='sigmoid') + binary_crossentropy
   Salida: [p1, p2, p3, ...] cada una independiente

REGRESIÓN:
   Dense(1, activation='linear') + mse/mae
   Salida: valor continuo


PREGUNTA 7: ¿Puedo usar embeddings pre-entrenados?
======================================================
SÍ! Muy buena idea:
   - Word2Vec, GloVe, FastText
   - Mejor rendimiento con menos datos
   - Transferencia de conocimiento

Ejemplo:
   embedding_layer = Embedding(
       vocab_size, 
       embedding_dim,
       weights=[embedding_matrix],  # Cargar pre-entrenados
       trainable=False  # No entrenar, solo usar
   )
"""

print("Ver comentarios arriba para respuestas detalladas")
print("\nPuntos clave para el examen:")
print("1. SIEMPRE usar oov_token")
print("2. GlobalAveragePooling > Flatten (menos parámetros)")
print("3. LSTM cuando el orden importa")
print("4. Capa de salida según tipo de problema")
print("5. Dropout y regularización contra overfitting")

## 9. EJEMPLO DE MEJORA: Comparación Práctica

In [ ]:
# Modelo BÁSICO (sin optimizar)
model_basic = keras.Sequential([
    layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    layers.Flatten(),
    layers.Dense(1, activation='sigmoid')
], name='Basic_Model')

model_basic.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Modelo MEJORADO (optimizado)
model_improved = keras.Sequential([
    layers.Embedding(vocab_size, 32, input_length=max_length),  # Más dimensiones
    layers.GlobalAveragePooling1D(),  # Mejor que Flatten
    layers.Dense(64, activation='relu'),  # Capa oculta más grande
    layers.Dropout(0.5),  # Regularización
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
], name='Improved_Model')

model_improved.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("MODELO BÁSICO:")
print(f"   Parámetros: {model_basic.count_params():,}")
model_basic.summary()

print("\n" + "=" * 60)
print("MODELO MEJORADO:")
print(f"   Parámetros: {model_improved.count_params():,}")
model_improved.summary()

print("\nMEJORAS IMPLEMENTADAS:")
print("- Embedding dimension: 16 -> 32 (más capacidad)")
print("- Flatten -> GlobalAveragePooling (menos parámetros)")
print("- Añadidas capas ocultas profundas")
print("- Dropout para regularización")
print("- Arquitectura más robusta")

## 10. CHEAT SHEET FINAL - Para el Examen

### Checklist Rápido:

```python
# 1. TOKENIZER
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")  # SIEMPRE oov_token

# 2. EMBEDDINGS
Embedding(vocab_size, embedding_dim, input_length=max_length)

# 3. REDUCCIÓN DE DIMENSIONALIDAD
GlobalAveragePooling1D()  # MEJOR que Flatten()
# o
LSTM(units, return_sequences=False)  # Para secuencias

# 4. REGULARIZACIÓN
Dropout(0.3-0.5)  # Después de capas densas

# 5. CAPA DE SALIDA - Según problema
# Binaria:      Dense(1, activation='sigmoid')
# Multiclase:   Dense(n, activation='softmax')
# Multi-label:  Dense(n, activation='sigmoid')
# Regresión:    Dense(1, activation='linear')

# 6. COMPILACIÓN - Según problema
# Binaria:      loss='binary_crossentropy'
# Multiclase:   loss='sparse_categorical_crossentropy'
# Multi-label:  loss='binary_crossentropy'
# Regresión:    loss='mse' o 'mae'
```

### Preguntas Frecuentes de Examen:

1. **"¿Qué cambiarías para mejorar este modelo?"**
   - Añadir Dropout
   - Cambiar Flatten por GlobalAveragePooling1D
   - Aumentar embedding_dim si hay datos
   - Usar LSTM si el contexto es importante
   - Añadir más capas ocultas

2. **"¿Qué pasa si no usas oov_token?"**
   - Palabras desconocidas se ignoran
   - Pérdida de información
   - Peor rendimiento en nuevos textos

3. **"¿Cuándo usar LSTM?"**
   - Textos largos
   - Orden de palabras importante
   - Dataset grande
   - Recursos computacionales disponibles

4. **"¿Cómo evitar overfitting?"**
   - Dropout
   - Reducir complejidad del modelo
   - Más datos de entrenamiento
   - Early stopping
   - Regularización L2

5. **"¿Qué capa de salida usar?"**
   - Ver tabla en sección 6

## RESUMEN EJECUTIVO PARA EL EXAMEN

### ERRORES COMUNES A EVITAR:
1. No usar `oov_token` en el Tokenizer
2. Usar Flatten cuando GlobalAveragePooling es mejor
3. Olvidar hacer `pad_sequences` 
4. Usar loss incorrecta para el tipo de problema
5. No normalizar/preprocesar textos
6. No usar Dropout (overfitting casi seguro)

### BUENAS PRÁCTICAS:
1. Tokenizer con `oov_token="<OOV>"`
2. GlobalAveragePooling1D para eficiencia
3. LSTM cuando el orden importa
4. Dropout(0.3-0.5) para regularización
5. Embedding_dim según tamaño del dataset
6. Capa de salida correcta según problema

### ARQUITECTURAS SEGÚN COMPLEJIDAD:

**Simple (Dataset pequeño):**
```
Embedding -> GlobalAveragePooling1D -> Dense -> Output
```

**Intermedio:**
```
Embedding -> GlobalAveragePooling1D -> Dense -> Dropout -> Dense -> Output
```

**Avanzado:**
```
Embedding -> LSTM/Bidirectional -> Dense -> Dropout -> Dense -> Output
```

---

**Suerte en el examen!**